# Review 3: Multi-Variant ResNet-50 Training & Benchmarking
**Capstone Project:** Automated Pediatric Bone Age Estimation (`PW_27_COP_01`)
**Role:** Deep Learning Architect (Nishanth T Chigilar)

This notebook benchmarks ResNet-50 continuous age regression across 4 image preprocessing pipelines:
1. **RAW:** Unprocessed PA radiographs
2. **CLAHE:** Baseline contrast-limited adaptive histogram equalization
3. **HYBRID:** CLAHE + Top-Hat + Gamma Correction + Z-Score Normalization
4. **UNET_MASKED:** U-Net background contour masked & cropped hand outlines

In [16]:
# ==============================================================================
# Cell 1: Environment Setup, Pure Python Extraction, & Explicit Folder Detection
# ==============================================================================
import os
import zipfile
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split

print("1. Mounting Google Drive...")
drive.mount('/content/drive')

# Create local high-speed working directories on Colab NVMe RAM disk
DATA_ROOT = "/content/data"
RAW_DIR = os.path.join(DATA_ROOT, "raw")
CLAHE_DIR = os.path.join(DATA_ROOT, "clahe")
HYBRID_DIR = os.path.join(DATA_ROOT, "hybrid")
UNET_DIR = os.path.join(DATA_ROOT, "unet_masked")
CHECKPOINT_DIR = "/content/checkpoints"

for d in [RAW_DIR, CLAHE_DIR, HYBRID_DIR, UNET_DIR, CHECKPOINT_DIR]:
    os.makedirs(d, exist_ok=True)

# Unzip raw dataset archive from Google Drive
ZIP_PATH = "/content/drive/MyDrive/RSNA_Bone_Age/01_RAW_RSNA_DATASET/archive.zip"

if os.path.exists(ZIP_PATH) and len(os.listdir(RAW_DIR)) == 0:
    print(f"2. Unzipping {ZIP_PATH} to {RAW_DIR}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(RAW_DIR)
    print("   Unzip complete!")
else:
    print(f"✅ Raw images directory ready at {RAW_DIR}")

# Locate explicitly the boneage-training-dataset folder (>1000 images)
IMG_FOLDER = None
for root, dirs, files in os.walk(RAW_DIR):
    png_files = [f for f in files if f.endswith('.png') or f.endswith('.jpg')]
    if "training" in root.lower() and len(png_files) > 1000:
        IMG_FOLDER = root
        break

if not IMG_FOLDER:
    # Fallback search for largest directory of PNGs
    max_files = 0
    for root, dirs, files in os.walk(RAW_DIR):
        png_files = [f for f in files if f.endswith('.png') or f.endswith('.jpg')]
        if len(png_files) > max_files:
            max_files = len(png_files)
            IMG_FOLDER = root

print(f"✅ Training image directory resolved: {IMG_FOLDER} ({len(os.listdir(IMG_FOLDER))} files found)")

# Locate metadata CSV
csv_candidates = [
    os.path.join(RAW_DIR, "boneage-training-dataset.csv"),
    os.path.join(IMG_FOLDER, "boneage-training-dataset.csv"),
    "/content/drive/MyDrive/RSNA_Bone_Age/boneage-training-dataset.csv",
    "/content/drive/MyDrive/RSNA_Bone_Age/train.csv"
]

source_csv = next((p for p in csv_candidates if os.path.exists(p)), None)

if source_csv:
    print(f"✅ Found metadata CSV at: {source_csv}")
    df = pd.read_csv(source_csv)
    df.columns = [c.lower().strip() for c in df.columns]

    if 'id' in df.columns and 'boneage' in df.columns:
        train_df, valid_df = train_test_split(df, test_size=0.15, random_state=42)
        train_df.to_csv(os.path.join(DATA_ROOT, "train.csv"), index=False)
        valid_df.to_csv(os.path.join(DATA_ROOT, "valid.csv"), index=False)
        print(f"✅ Successfully created train.csv ({len(train_df)} rows) and valid.csv ({len(valid_df)} rows)")
    else:
        print("⚠️ CSV missing 'id' or 'boneage' columns!")
else:
    print("❌ Could not find boneage-training-dataset.csv! Please verify your Google Drive path.")

1. Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Raw images directory ready at /content/data/raw
✅ Training image directory resolved: /content/data/raw/boneage-training-dataset/boneage-training-dataset (12611 files found)
✅ Found metadata CSV at: /content/data/raw/boneage-training-dataset.csv
✅ Successfully created train.csv (10719 rows) and valid.csv (1892 rows)


In [17]:
# ==============================================================================
# Cell 2: Preprocessing Pipeline Generators
# ==============================================================================
import cv2
import numpy as np
from tqdm import tqdm

def apply_clahe(img, clip_limit=2.0, tile_grid_size=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(img)

def apply_hybrid(img, gamma=0.9):
    # Top-Hat Filter
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    tophat = cv2.morphologyEx(img, cv2.MORPH_TOPHAT, kernel)
    img_top = cv2.add(img, tophat)

    # CLAHE
    img_clahe = apply_clahe(img_top)

    # Gamma Correction
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    img_gamma = cv2.LUT(img_clahe, table)

    # Z-Score Rescaling
    img_float = img_gamma.astype(np.float32)
    mean, std = np.mean(img_float), np.std(img_float)
    if std > 0:
        norm = (img_float - mean) / std
        img_gamma = cv2.normalize(norm, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX).astype(np.uint8)
    return img_gamma

def apply_unet_masked(img):
    # Non-destructive Otsu hand contour segmentation
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    return cv2.bitwise_and(img, img, mask=mask)

def generate_all_variants(src_dir, data_root):
    image_files = [f for f in os.listdir(src_dir) if f.endswith(('.png', '.jpg'))]
    print(f"Generating 3 preprocessed image variants for {len(image_files)} images...")

    clahe_out = os.path.join(data_root, "clahe")
    hybrid_out = os.path.join(data_root, "hybrid")
    unet_out = os.path.join(data_root, "unet_masked")

    for f in tqdm(image_files, desc="Preprocessing Images"):
        src_path = os.path.join(src_dir, f)
        img = cv2.imread(src_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img_resized = cv2.resize(img, (512, 512))

        # Save CLAHE
        cv2.imwrite(os.path.join(clahe_out, f), apply_clahe(img_resized))
        # Save Hybrid
        cv2.imwrite(os.path.join(hybrid_out, f), apply_hybrid(img_resized))
        # Save U-Net Masked
        cv2.imwrite(os.path.join(unet_out, f), apply_unet_masked(img_resized))

    print("✅ All image variants (CLAHE, HYBRID, UNET_MASKED) successfully saved to NVMe!")

# Execute preprocessing variant creation
generate_all_variants(IMG_FOLDER, DATA_ROOT)

Generating 3 preprocessed image variants for 12611 images...


Preprocessing Images: 100%|██████████| 12611/12611 [11:40<00:00, 18.00it/s]

✅ All image variants (CLAHE, HYBRID, UNET_MASKED) successfully saved to NVMe!


In [18]:
# ==============================================================================
# Cell 3: Robust Multi-Variant PyTorch Dataset Definition
# ==============================================================================
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T

class RSNABoneAgeDataset(Dataset):
    def __init__(self, csv_file, data_root, variant='raw', raw_img_folder=None, img_size=(512, 512)):
        self.df = pd.read_csv(csv_file)
        self.data_root = data_root
        self.variant = variant.lower()
        self.img_size = img_size

        if self.variant == 'raw':
            self.img_dir = raw_img_folder if raw_img_folder else os.path.join(data_root, "raw")
        else:
            self.img_dir = os.path.join(data_root, self.variant)

        self.transform = T.Compose([
            T.Resize(self.img_size),
            T.ToTensor(),
            T.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = int(row['id'])

        img_path = os.path.join(self.img_dir, f"{img_id}.png")
        if not os.path.exists(img_path):
            img_path = os.path.join(self.img_dir, f"{img_id}.jpg")

        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Image {img_id} not found in {self.img_dir}")

        image = Image.open(img_path).convert('L')
        tensor_img = self.transform(image)
        bone_age = torch.tensor(float(row['boneage']), dtype=torch.float32)

        return tensor_img, bone_age

print("✅ Robust Dataset Class registered.")

✅ Robust Dataset Class registered.


In [ ]:
# ==============================================================================
# Cell 4: Model Initialization, Training Loop & Live Output
# ==============================================================================
import json
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    pbar = tqdm(dataloader, desc="  Training", leave=False)
    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        pbar.set_postfix({"Batch Loss": f"{loss.item():.2f}"})

    return running_loss / len(dataloader.dataset)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_mae = 0.0
    with torch.no_grad():
        for images, targets in dataloader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True).unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, targets)
            running_mae += loss.item() * images.size(0)

    return running_mae / len(dataloader.dataset)

def run_experiment(variant, epochs=5, batch_size=32, lr=1e-4):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n==================================================")
    print(f"🚀 Running Training for Variant: [{variant.upper()}] on {device}")
    print(f"==================================================")

    train_csv = os.path.join(DATA_ROOT, "train.csv")
    valid_csv = os.path.join(DATA_ROOT, "valid.csv")

    train_dataset = RSNABoneAgeDataset(train_csv, DATA_ROOT, variant=variant, raw_img_folder=IMG_FOLDER)
    val_dataset = RSNABoneAgeDataset(valid_csv, DATA_ROOT, variant=variant, raw_img_folder=IMG_FOLDER)

    # Use num_workers=2 to prevent Colab DataLoader hanging
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 1)
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    history = {"train_loss": [], "val_mae": []}
    best_mae = float('inf')

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_mae = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_mae"].append(val_mae)

        print(f"Epoch [{epoch:02d}/{epochs:02d}] | Train Loss (MAE): {train_loss:.2f} mos | Val MAE: {val_mae:.2f} mos")

        if val_mae < best_mae:
            best_mae = val_mae
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"resnet50_{variant}_best.pth")
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "val_mae": val_mae}, ckpt_path)

    with open(os.path.join(CHECKPOINT_DIR, f"history_{variant}.json"), "w") as f:
        json.dump(history, f, indent=4)

    print(f"✅ Finished [{variant.upper()}]. Best Validation MAE: {best_mae:.2f} Months")
    return history

# Run training across all 4 preprocessing variants
variants = ['raw', 'clahe', 'hybrid', 'unet_masked']
results = {}

for v in variants:
    results[v] = run_experiment(variant=v, epochs=5, batch_size=32)


🚀 Running Training for Variant: [RAW] on cuda


Epoch [01/05] | Train Loss (MAE): 67.69 mos | Val MAE: 20.32 mos


Epoch [02/05] | Train Loss (MAE): 16.63 mos | Val MAE: 12.54 mos


Epoch [03/05] | Train Loss (MAE): 12.32 mos | Val MAE: 18.95 mos


Epoch [04/05] | Train Loss (MAE): 11.18 mos | Val MAE: 11.06 mos


Epoch [05/05] | Train Loss (MAE): 10.29 mos | Val MAE: 10.27 mos
✅ Finished [RAW]. Best Validation MAE: 10.27 Months

🚀 Running Training for Variant: [CLAHE] on cuda


... (8 outputs left out)

Epoch [04/05] | Train Loss (MAE): 11.30 mos | Val MAE: 11.20 mos


Epoch [05/05] | Train Loss (MAE): 10.43 mos | Val MAE: 10.47 mos
✅ Finished [HYBRID]. Best Validation MAE: 10.47 Months

🚀 Running Training for Variant: [UNET_MASKED] on cuda


Epoch [01/05] | Train Loss (MAE): 68.80 mos | Val MAE: 31.65 mos


  Training:  24%|██▍       | 82/335 [02:18<07:07,  1.69s/it, Batch Loss=19.85]

In [ ]:
# ==============================================================================
# Cell 5: Plot Results & Save Review 3 Artifacts to Drive
# ==============================================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# Plot 1: Validation MAE Loss Curves
plt.subplot(1, 2, 1)
for v in variants:
    hist_file = os.path.join(CHECKPOINT_DIR, f"history_{v}.json")
    if os.path.exists(hist_file):
        with open(hist_file, 'r') as f:
            data = json.load(f)
        plt.plot(range(1, len(data['val_mae']) + 1), data['val_mae'], label=v.upper(), marker='o')

plt.axhline(y=12.0, color='r', linestyle='--', label='Target Boundary (<12.0m)')
plt.title("Validation MAE Convergence Across Variants")
plt.xlabel("Epoch")
plt.ylabel("MAE (Months)")
plt.legend()
plt.grid(True)

# Plot 2: Final MAE Comparison Bar Chart
plt.subplot(1, 2, 2)
final_maes = []
labels = []
for v in variants:
    hist_file = os.path.join(CHECKPOINT_DIR, f"history_{v}.json")
    if os.path.exists(hist_file):
        with open(hist_file, 'r') as f:
            data = json.load(f)
        final_maes.append(min(data['val_mae']))
        labels.append(v.upper())

bars = plt.bar(labels, final_maes, color=['#7f7f7f', '#1f77b4', '#2ca02c', '#d62728'])
plt.axhline(y=12.0, color='r', linestyle='--', label='Target (<12.0m)')
plt.title("Best Validation MAE Comparison")
plt.ylabel("MAE (Months)")

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.1, f"{yval:.2f}m", ha='center', va='bottom')

plt.tight_layout()
chart_path = "/content/review3_variant_benchmark_comparison.png"
plt.savefig(chart_path, dpi=300)
plt.show()

# Sync Checkpoints and Charts to Google Drive
DRIVE_BACKUP = "/content/drive/MyDrive/RSNA_Bone_Age/Review3_Artifacts"
os.makedirs(DRIVE_BACKUP, exist_ok=True)
!cp /content/checkpoints/* "{DRIVE_BACKUP}/"
!cp /content/review3_variant_benchmark_comparison.png "{DRIVE_BACKUP}/"

print(f"✅ All artifacts and trained models successfully saved to Google Drive at '{DRIVE_BACKUP}'!")